## Text chunked and the check the quality of chunk

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## Setup configuration

In [2]:
class Config:
    # setup mistral configuration
    mistral_api_key = os.getenv("MISTRAL_API_KEY")
    mistral_chat_model = os.getenv("MISTRAL_CHAT_MODEL")
    mistral_embed_model = "mistral-embed"
    mistral_embed_dimension =  int(os.getenv("MISTRAL_EMBED_DIMENSION")) or 1024

## Setup LLM Service

In [3]:
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings

class MistralService:
    chatModel : ChatMistralAI
    embeddingModel : MistralAIEmbeddings

    def __init__(self):
        self.chatModel = self.connectMistralChatModel()
        self.embeddingModel = self.connectMistralEmbedModel()

    def connectMistralChatModel(self, model_name : str = Config.mistral_chat_model) -> ChatMistralAI :
        try:
            return ChatMistralAI(
                api_key = Config.mistral_api_key,
                model = model_name
            )
        except Exception as e:
            print(f"Something went wrong in the {model_name} connection...")
            raise
        
    def connectMistralEmbedModel(self, model_name: str = Config.mistral_embed_model) -> MistralAIEmbeddings :
        try:
            return MistralAIEmbeddings(
                api_key = Config.mistral_api_key,
                model = model_name
            )
        except Exception as e:
            print(f"Something went wrong in the {model_name} connection...")
            raise
    
    def getChatModel(self) -> ChatMistralAI:
        return self.chatModel
    
    def getEmbedModel(self) -> MistralAIEmbeddings:
        return self.embeddingModel

## Setup mock document data

In [39]:
from langchain_core.documents import Document
from datetime import datetime

ts = datetime.now().isoformat()

document_list = [
    # --- HIGH VALUE DOCUMENTS (Rich Content) ---
    Document(
        metadata={'document_name': 'ai_ethics.pdf', 'document_created_at': ts},
        page_content="Ethical risks in AI primarily involve algorithmic bias. If a model is trained on biased data, it will reproduce those inequities in its predictions, affecting hiring and credit scoring."
    ),
    Document(
        metadata={'document_name': 'deep_learning.pdf', 'document_created_at': ts},
        page_content="Backpropagation is the central mechanism by which neural networks learn. It calculates the gradient of the loss function with respect to the weights by applying the chain rule."
    ),
    Document(
        metadata={'document_name': 'nlp_basics.pdf', 'document_created_at': ts},
        page_content="Transformers revolutionized NLP by using self-attention mechanisms, allowing the model to weigh the importance of different words in a sentence regardless of their distance."
    ),
    Document(
        metadata={'document_name': 'robotics.pdf', 'document_created_at': ts},
        page_content="Computer vision enables autonomous vehicles to interpret their surroundings. Sensors like LiDAR and cameras work together to create a 3D map for real-time navigation."
    ),
    Document(
        metadata={'document_name': 'data_science.pdf', 'document_created_at': ts},
        page_content="Data cleaning is often 80% of the work in ML. Removing outliers, handling missing values, and normalizing features are critical steps before training any model."
    ),

    # --- LOW VALUE DOCUMENTS (Noisy/Empty/Irrelevant) ---
    Document(
        metadata={'document_name': 'corrupted.pdf', 'document_created_at': ts},
        page_content=" 12345 ##### error_code: 0x000921 --- standard system failure. [REDACTED]"
    ),
    Document(
        metadata={'document_name': 'placeholder.pdf', 'document_created_at': ts},
        page_content="This page intentionally left blank. Please ignore this text for analysis purposes."
    ),
    Document(
        metadata={'document_name': 'random_chars.txt', 'document_created_at': ts},
        page_content="asdfghjkl; qwertyuiop 1234567890 !@#$%^&*()_+"
    )
]

## Document chunk validity checker

In [8]:
from pydantic import BaseModel, Field

class ChunkQualityOutput(BaseModel):
    quality_score : float = Field(..., ge= 0, le=1 )

In [44]:
from langchain_core.prompts import PromptTemplate

prompt_message = """
    Role:
    You are a Content Quality Auditor specialized in Data Pre-processing for RAG systems.
    
    Task:
    Your task is the generate the value how much that statement quality is ? is that wanted valud content or its just normal statement. Evaluate the "Informational Density" of the provided statement. Your goal is to distinguish between high-value knowledge and low-value structural noise.

    statement:
    {statement}
"""

prompt_template = PromptTemplate.from_template(prompt_message)

def pareprePrompt(content: str) -> PromptTemplate:
    return prompt_template.invoke({
        "statement" : content
    })

In [45]:
class DocumentChunkQualityManager:
    qualifiedDocuments : list[Document]

    def __init__(self, docs: list[Document]):
        self.qualifiedDocuments = []
        self.__getMistralService()
        self.qualifiedQualityChunks(docs)
        
    def __getMistralService(self):
        mistral = MistralService()
        llm = mistral.getChatModel()
        self.structured_llm = llm.with_structured_output(ChunkQualityOutput)

    def getChunkQuality(self, doc: Document):
        prompt = pareprePrompt(doc.page_content)
        chunk_quality_repsponse = structured_llm.invoke(prompt)

        doc.metadata['chunk_quality'] = chunk_quality_repsponse.quality_score
        return chunk_quality_repsponse.quality_score

    def qualifiedQualityChunks(self, docs: list[Document]):
        for doc in docs:
            quality_score = self.getChunkQuality(doc)

            if quality_score > 0.6:
                self.qualifiedDocuments.append(doc)
        
        for i, doc in enumerate(docs, start= 1):
            doc.metadata['chunk_arranged_number'] = i

        return self.qualifiedDocuments

    def getQualifiedChunks(self):
        return self.qualifiedDocuments


## Work on document validator

In [46]:
chunkQualityManager = DocumentChunkQualityManager(document_list)
qualified_docs = chunkQualityManager.getQualifiedChunks()

print(qualified_docs)

[Document(metadata={'document_name': 'ai_ethics.pdf', 'document_created_at': '2026-03-13T23:47:02.724754', 'chunk_quality': 0.95, 'chunk_arranged_number': 1}, page_content='Ethical risks in AI primarily involve algorithmic bias. If a model is trained on biased data, it will reproduce those inequities in its predictions, affecting hiring and credit scoring.'), Document(metadata={'document_name': 'deep_learning.pdf', 'document_created_at': '2026-03-13T23:47:02.724754', 'chunk_quality': 0.95, 'chunk_arranged_number': 2}, page_content='Backpropagation is the central mechanism by which neural networks learn. It calculates the gradient of the loss function with respect to the weights by applying the chain rule.'), Document(metadata={'document_name': 'nlp_basics.pdf', 'document_created_at': '2026-03-13T23:47:02.724754', 'chunk_quality': 0.95, 'chunk_arranged_number': 3}, page_content='Transformers revolutionized NLP by using self-attention mechanisms, allowing the model to weigh the importanc

## Analyze the results

In [47]:
print(f"Total document count : {len(document_list)}")
print(f"Total qualified document count : {len(qualified_docs)}")

Total document count : 8
Total qualified document count : 5
